# Lab 2 – Task 2.1: exploring the raw downloads

Topic: **air pollution and respiratory disease**. For every source this notebook

1. calls the extractor, which downloads the data and saves the raw response in `data/raw/<source>/`,
2. transforms the saved file into the normalized table (`country_iso3`, `indicator_code`, `year`, `sex`, `value`; WHO also has `residence_area`, `value_low` and `value_high`),
3. inspects the result with `df.head()`, `df.info()` and `df.isna().sum()`,

and ends with the share of missing values per indicator. Nothing is cleaned or stored here:
that is the job of `etl/clean.py` and the import scripts.

Running the notebook needs internet access (about a minute). If a source cannot be reached,
the error is printed and the file saved by an earlier run is used, if there is one.

In [1]:
import sys
from functools import partial
from pathlib import Path

import pandas as pd
import requests

# The notebook lives in notebooks/; the project (with the etl package) is one folder up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from etl import import_owid, import_who, import_worldbank
from etl.extract_owid import download_owid_chart
from etl.extract_who import WhoApiError, download_who_indicator
from etl.extract_worldbank import (
    WorldBankApiError,
    download_worldbank_countries,
    download_worldbank_indicator,
)
from etl.transform_owid import transform_owid_chart_file
from etl.transform_who import transform_who_file
from etl.transform_worldbank import (
    transform_worldbank_country_file,
    transform_worldbank_indicator_file,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 40)

In [2]:
def extract(label, download, cached):
    """Run one extractor; if the source cannot be reached, use the file saved earlier."""
    try:
        path, url = download()
        print(f"downloaded {label}\n  from  {url}\n  saved {path.relative_to(ROOT)}")
        return path
    except (requests.RequestException, WhoApiError, WorldBankApiError) as exc:
        print(f"FAILED {label}: {type(exc).__name__}: {exc}")
        if cached.exists():
            print(f"  using the file saved earlier: {cached.relative_to(ROOT)}")
            return cached
        return None


def inspect(frame, label):
    """df.head(), df.info() and df.isna().sum() of one normalized table."""
    print(f"=== {label}: {len(frame)} rows")
    display(frame.head())
    frame.info()
    display(frame.isna().sum().to_frame("missing values").T)


normalized = {}  # indicator code -> normalized table

## World Bank

The country list first (it becomes the `country` table), then the five indicators for every
economy and the years 2000–2024. The API returns one record per economy and year, with
`value: null` where there is no data, and mixes countries with aggregates such as `WLD`.

In [3]:
countries_path = extract(
    "World Bank countries",
    download_worldbank_countries,
    import_worldbank.RAW_DIR / "countries.json",
)
countries = transform_worldbank_country_file(countries_path)
valid_iso3 = set(countries["iso3"])
inspect(countries, "World Bank countries (aggregates removed)")
countries["region"].value_counts()

downloaded World Bank countries
  from  https://api.worldbank.org/v2/country?per_page=400&format=json
  saved data/raw/worldbank/countries.json
=== World Bank countries (aggregates removed): 217 rows


,iso3,iso2,name,region,income_group
0,ABW,AW,Aruba,Latin America & Caribbean,High income
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanist...",Low income
2,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income
3,ALB,AL,Albania,Europe & Central Asia,Upper middle income
4,AND,AD,Andorra,Europe & Central Asia,High income


<class 'pandas.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   iso3          217 non-null    str  
 1   iso2          217 non-null    str  
 2   name          217 non-null    str  
 3   region        217 non-null    str  
 4   income_group  217 non-null    str  
dtypes: str(5)
memory usage: 8.6 KB


,iso3,iso2,name,region,income_group
missing values,0,0,0,0,0


region
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64

In [4]:
for spec in import_worldbank.INDICATORS:
    path = extract(
        spec.code,
        partial(download_worldbank_indicator, spec.code, import_worldbank.YEARS),
        import_worldbank.RAW_DIR / spec.filename,
    )
    if path:
        normalized[spec.code] = transform_worldbank_indicator_file(path, spec.code)
        inspect(normalized[spec.code], f"{spec.code} – {spec.name} ({spec.unit})")
        print()

downloaded EN.ATM.PM25.MC.M3
  from  https://api.worldbank.org/v2/country/all/indicator/EN.ATM.PM25.MC.M3?date=2000%3A2024&per_page=10000&format=json
  saved data/raw/worldbank/EN.ATM.PM25.MC.M3_2000-2024.json
=== EN.ATM.PM25.MC.M3 – PM2.5 air pollution, mean annual exposure (µg/m³): 6650 rows


,country_iso3,indicator_code,year,sex,value
0,AFE,EN.ATM.PM25.MC.M3,2024,None,NaN
1,AFE,EN.ATM.PM25.MC.M3,2023,None,25.283403
2,AFE,EN.ATM.PM25.MC.M3,2022,None,26.383916
3,AFE,EN.ATM.PM25.MC.M3,2021,None,25.529458
4,AFE,EN.ATM.PM25.MC.M3,2020,None,25.660129


<class 'pandas.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6650 non-null   str    
 1   indicator_code  6650 non-null   str    
 2   year            6650 non-null   str    
 3   sex             0 non-null      object 
 4   value           5952 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 259.9+ KB


,country_iso3,indicator_code,year,sex,value
missing values,0,0,0,6650,698


downloaded SH.STA.AIRP.P5
  from  https://api.worldbank.org/v2/country/all/indicator/SH.STA.AIRP.P5?date=2000%3A2024&per_page=10000&format=json
  saved data/raw/worldbank/SH.STA.AIRP.P5_2000-2024.json
=== SH.STA.AIRP.P5 – Mortality rate attributed to household and ambient air pollution, age-standardized (per 100,000 population): 6650 rows


,country_iso3,indicator_code,year,sex,value
0,AFE,SH.STA.AIRP.P5,2024,None,NaN
1,AFE,SH.STA.AIRP.P5,2023,None,NaN
2,AFE,SH.STA.AIRP.P5,2022,None,NaN
3,AFE,SH.STA.AIRP.P5,2021,None,NaN
4,AFE,SH.STA.AIRP.P5,2020,None,NaN


... (10 outputs left out)

downloaded NY.GDP.PCAP.CD
  from  https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?date=2000%3A2024&per_page=10000&format=json
  saved data/raw/worldbank/NY.GDP.PCAP.CD_2000-2024.json
=== NY.GDP.PCAP.CD – GDP per capita (current US$): 6650 rows


,country_iso3,indicator_code,year,sex,value
0,AFE,NY.GDP.PCAP.CD,2024,None,1628.227289
1,AFE,NY.GDP.PCAP.CD,2023,None,1571.132704
2,AFE,NY.GDP.PCAP.CD,2022,None,1675.902524
3,AFE,NY.GDP.PCAP.CD,2021,None,1560.894626
4,AFE,NY.GDP.PCAP.CD,2020,None,1351.503167


<class 'pandas.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6650 non-null   str    
 1   indicator_code  6650 non-null   str    
 2   year            6650 non-null   str    
 3   sex             0 non-null      object 
 4   value           6442 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 259.9+ KB


,country_iso3,indicator_code,year,sex,value
missing values,0,0,0,6650,208


## WHO Global Health Observatory

PM2.5 concentration by area of residence (urban, rural, total) with its uncertainty
interval, and the death rate attributed to air pollution by sex.

In [5]:
for spec in import_who.INDICATORS:
    path = extract(
        spec.code,
        partial(download_who_indicator, spec.code, import_who.YEAR_FROM),
        import_who.RAW_DIR / spec.filename,
    )
    if path:
        normalized[spec.code] = transform_who_file(path, spec.code)
        inspect(normalized[spec.code], f"{spec.code} – {spec.name} ({spec.unit})")
        print()
    else:
        print(f"no {spec.code} data to inspect\n")

FAILED SDGPM25: HTTPError: 503 Server Error: Service Unavailable for url: https://ghoapi.azureedge.net/api/SDGPM25?%24filter=SpatialDimType+eq+%27COUNTRY%27+and+TimeDim+ge+2000
no SDGPM25 data to inspect



FAILED SDGAIRBODA: HTTPError: 503 Server Error: Service Unavailable for url: https://ghoapi.azureedge.net/api/SDGAIRBODA?%24filter=SpatialDimType+eq+%27COUNTRY%27+and+TimeDim+ge+2000
no SDGAIRBODA data to inspect



## Our World in Data

One CSV file per chart (`entity`, `code`, `year` and one value column). Only rows with a
value are in the file, so here missing data shows up as missing country-years, not as
empty values.

In [6]:
for spec, slug in zip(import_owid.CHARTS, import_owid.SLUGS):
    path = extract(
        spec.code,
        partial(download_owid_chart, slug),
        import_owid.RAW_DIR / spec.filename,
    )
    if path:
        normalized[spec.code] = transform_owid_chart_file(path, spec.code)
        inspect(normalized[spec.code], f"{spec.code} – {spec.name} ({spec.unit})")
        print()

downloaded OWID_CRD_DEATH_RATE
  from  https://ourworldindata.org/grapher/chronic-respiratory-diseases-death-rate-who-mdb.csv?v=1&csvType=full&useColumnShortNames=true
  saved data/raw/owid/chronic-respiratory-diseases-death-rate-who-mdb_full.csv
=== OWID_CRD_DEATH_RATE – Chronic respiratory diseases death rate, age-standardized (WHO Mortality Database) (deaths per 100,000 people): 4939 rows


,country_iso3,indicator_code,year,sex,value
0,ALB,OWID_CRD_DEATH_RATE,1987,None,26.313232
1,ALB,OWID_CRD_DEATH_RATE,1988,None,26.932293
2,ALB,OWID_CRD_DEATH_RATE,1989,None,25.726141
3,ALB,OWID_CRD_DEATH_RATE,1992,None,21.516480
4,ALB,OWID_CRD_DEATH_RATE,1993,None,23.432169


<class 'pandas.DataFrame'>
RangeIndex: 4939 entries, 0 to 4938
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    4939 non-null   str    
 1   indicator_code  4939 non-null   str    
 2   year            4939 non-null   int64  
 3   sex             0 non-null      object 
 4   value           4939 non-null   float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 193.1+ KB


,country_iso3,indicator_code,year,sex,value
missing values,0,0,0,4939,0


downloaded OWID_PM25_EXPOSURE
  from  https://ourworldindata.org/grapher/outdoor-air-pollution-exposure.csv?v=1&csvType=full&useColumnShortNames=true
  saved data/raw/owid/outdoor-air-pollution-exposure_full.csv
=== OWID_PM25_EXPOSURE – Outdoor air pollution exposure, population-weighted PM2.5 (SatPM) (µg/m³): 6912 rows


,country_iso3,indicator_code,year,sex,value
0,AFG,OWID_PM25_EXPOSURE,1998,None,20.7
1,AFG,OWID_PM25_EXPOSURE,1999,None,23.0
2,AFG,OWID_PM25_EXPOSURE,2000,None,30.4
3,AFG,OWID_PM25_EXPOSURE,2001,None,35.0
4,AFG,OWID_PM25_EXPOSURE,2002,None,33.9


<class 'pandas.DataFrame'>
RangeIndex: 6912 entries, 0 to 6911
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6912 non-null   str    
 1   indicator_code  6912 non-null   str    
 2   year            6912 non-null   int64  
 3   sex             0 non-null      object 
 4   value           6912 non-null   float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 270.1+ KB


,country_iso3,indicator_code,year,sex,value
missing values,0,0,0,6912,0


## Share of missing values per indicator

Two measures, because the sources report missing data differently:

* **empty values in the file** – rows whose `value` is empty (World Bank `null`, WHO without `NumericValue`);
* **country-years without a value** – of all combinations of the World Bank countries and the
  years 2000–2024 (the study window), the share with no value. Only the total
  (both sexes, all areas) is counted, so the sources can be compared.

In [7]:
YEARS = range(2000, 2025)
grid_size = len(valid_iso3) * len(YEARS)


def missing_summary(code, frame):
    year = pd.to_numeric(frame["year"], errors="coerce")
    total = frame["sex"].isna() | frame["sex"].eq("SEX_BTSX")
    if "residence_area" in frame:
        total &= frame["residence_area"].isna() | frame["residence_area"].eq(
            "RESIDENCEAREATYPE_TOTL"
        )
    with_value = frame[
        total
        & frame["value"].notna()
        & frame["country_iso3"].isin(valid_iso3)
        & year.between(YEARS.start, YEARS.stop - 1)
    ]
    covered = with_value[["country_iso3", "year"]].drop_duplicates()
    data_years = year[frame["value"].notna()]
    return {
        "indicator": code,
        "rows in file": len(frame),
        "empty values": int(frame["value"].isna().sum()),
        "empty values %": round(100 * frame["value"].isna().mean(), 1),
        "countries with data": with_value["country_iso3"].nunique(),
        "years with data": (
            f"{int(data_years.min())}–{int(data_years.max())}" if len(data_years) else "–"
        ),
        "country-years without a value %": round(100 * (1 - len(covered) / grid_size), 1),
    }


missing = pd.DataFrame([missing_summary(c, f) for c, f in normalized.items()]).set_index(
    "indicator"
)
print(f"{len(valid_iso3)} countries x {len(YEARS)} years = {grid_size} country-years")
missing

217 countries x 25 years = 5425 country-years


,rows in file,empty values,empty values %,countries with data,years with data,country-years without a value %
indicator,,,,,,
EN.ATM.PM25.MC.M3,6650,698,10.5,200,2000–2023,11.5
SH.STA.AIRP.P5,6650,6419,96.5,183,2019–2019,96.6
SH.PRV.SMOK,6650,4733,71.2,166,2000–2024,72.5
EG.CFT.ACCS.ZS,6650,650,9.8,192,2000–2024,11.5
NY.GDP.PCAP.CD,6650,208,3.1,214,2000–2024,3.4
OWID_CRD_DEATH_RATE,4939,0,0.0,111,1950–2024,59.7
OWID_PM25_EXPOSURE,6912,0,0.0,215,1998–2024,0.9


## Notes

Run on 4 November 2026.

* **Downloads.** The World Bank (country list and 5 indicators) and Our World in Data (2 charts) answered normally. The WHO GHO API answered **HTTP 503 Service Unavailable** for both indicators, so there is no WHO table to inspect. The WHO import skips them until the API is back.
* **World Bank: a full grid with `null` values.** There is one record per economy and year: 266 economies × 25 years = 6,650 rows, of which 49 economies are aggregates such as `WLD` (removed by the cleaning). How much is missing reflects how each indicator is produced:
  * PM2.5: 10.5 % empty; the GBD model ends in 2023.
  * Air pollution mortality: 96.5 % empty; estimated for 2019 only.
  * Tobacco use: 71.2 % empty; estimates exist for 9 years only.
  * Clean cooking: 9.8 % empty.
  * GDP per capita: 3.1 % empty.
* **Our World in Data: only the rows with a value.** The files have 0 % empty values; missing data shows up as absent country-years instead. The chronic respiratory diseases death rate covers only 111 of the 217 countries, the ones in the WHO Mortality Database: 59.7 % of the country-years 2000–2024 have no value, and coverage falls in recent years. The satellite-based PM2.5 exposure covers almost every country (0.9 %).
* **Types.** World Bank years arrive as text (`"2019"`) and OWID codes include aggregates (`OWID_WRL`) and `OWID_KOS` for Kosovo. The cleaning handles these with `pd.to_numeric()`, the country filter and a code alias.
* **Consequence for the analysis.** The respiratory death rate is the bottleneck. Its gaps are concentrated in Sub-Saharan Africa and Asia, the regions with the highest PM2.5 exposure (see `docs/lab2_results.md`, check 7).